# 🎬 VideoClip Creator IA — TODO EN UNO (VERSIÓN ARREGLADA)

**Qué es:** Orquestador COMPLETO + LTX-Video en un solo Colab. Sin VPS, sin API keys obligatorias.

**Cómo usar:**
1. Menú `Entorno de ejecución → Cambiar tipo de entorno → GPU (T4)`
2. `Ctrl+F9` (Ejecutar todas)
3. Al final verás una URL grande `https://....trycloudflare.com` → cópiala
4. Pégala en el Lanzador (campo "URL del Orquestador") junto con el token

> ⏱ La primera ejecución tarda 8-12 min (instala todo + descarga el modelo).
> NO cierres esta pestaña mientras generas videoclips.

In [ ]:
# ⚙️ CELDA 1/5: Instalar TODO (5-8 min la 1ª vez)
!pip install -q fastapi "uvicorn[standard]" nest-asyncio python-multipart requests
!pip install -q librosa soundfile numpy
!pip install -q diffusers transformers accelerate sentencepiece imageio imageio-ffmpeg pillow
!apt-get update -qq && apt-get install -y -qq ffmpeg > /dev/null 2>&1
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared
!chmod +x /content/cloudflared
print('✅ TODO instalado (incluye python-multipart y ffmpeg)')

In [ ]:
# 🔐 CELDA 2/5: Configuración
# Token que también usa el Lanzador (debe ser EL MISMO en ambos sitios):
VCC_TOKEN = "5a9fba8ba6a04cbbae2a77bd720c0409"

# OPCIONAL: tu Gemini API key (empieza por AIzaSy...). Vacío = usa Pollinations Text (gratis, sin key)
GEMINI_API_KEY = ""

# 🎬 MODO DE VÍDEO: "kenburns" (rápido, SIN RAM, garantizado) o "ltx" (IA real, pesado)
MODO_VIDEO = "kenburns"

print('✅ Config OK' if len(VCC_TOKEN) > 10 else '⚠️ Pon un token')
print('   IA de guion:', 'Gemini' if GEMINI_API_KEY else 'Pollinations Text (gratis)')


In [ ]:
# 🤖 CELDA 3/5: Modelo de vídeo (solo si MODO_VIDEO == "ltx")
import torch, gc
pipe = None
if MODO_VIDEO == "ltx":
    from diffusers import LTXImageToVideoPipeline
    pipe = LTXImageToVideoPipeline.from_pretrained("Lightricks/LTX-Video", torch_dtype=torch.bfloat16)
    pipe.enable_sequential_cpu_offload(); gc.collect(); torch.cuda.empty_cache()
    print("✅ LTX-Video cargado (modo IA). GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "⚠️ SIN GPU")
else:
    print("✅ MODO KEN BURNS activo (zoom con FFmpeg). Sin modelo pesado → SIN riesgo de RAM. Listo.")


In [ ]:
# 🖥️ CELDA 4/5: SERVIDOR COMPLETO — todos los endpoints del orquestador
import base64, io, json, re, subprocess, threading, time, uuid, urllib.parse
from pathlib import Path
import nest_asyncio, requests, uvicorn, torch
from fastapi import FastAPI, Form, Header, HTTPException, UploadFile
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import FileResponse
from PIL import Image
from diffusers.utils import export_to_video

PROY = Path('/content/proyectos'); PROY.mkdir(exist_ok=True)

app = FastAPI(title='VideoClip Creator — Todo en Colab')
# CORS ABIERTO: imprescindible, el Lanzador está en otro dominio
app.add_middleware(CORSMiddleware, allow_origins=['*'], allow_methods=['*'], allow_headers=['*'])

def auth(t):
    if t != VCC_TOKEN: raise HTTPException(401, 'token inválido')

def carpeta(pid):
    if not pid.replace('-', '').isalnum(): raise HTTPException(400, 'id inválido')
    p = PROY / pid
    if not p.exists(): raise HTTPException(404, 'proyecto no existe')
    return p

def cargar(pid): return json.loads((carpeta(pid) / 'estado.json').read_text(encoding='utf-8'))
def guardar(pid, e): (carpeta(pid) / 'estado.json').write_text(json.dumps(e, indent=2, ensure_ascii=False), encoding='utf-8')

def _json(txt):
    txt = txt.strip()
    if txt.startswith('```'):
        txt = txt.split(chr(10), 1)[1] if chr(10) in txt else txt[3:]
    txt = txt.rstrip('`').strip()
    try: return json.loads(txt)
    except Exception:
        m = re.search(r'\{.*\}', txt, re.DOTALL)
        if m: return json.loads(m.group(0))
        raise HTTPException(500, 'La IA no devolvió JSON válido')

def ia_texto(prompt):
    if GEMINI_API_KEY:
        try:
            r = requests.post(
                'https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent?key=' + GEMINI_API_KEY,
                json={'contents': [{'parts': [{'text': prompt}]}],
                      'generationConfig': {'response_mime_type': 'application/json'}}, timeout=120)
            if r.status_code == 200:
                return _json(r.json()['candidates'][0]['content']['parts'][0]['text'])
            print('  ⚠️ Gemini', r.status_code, '→ uso Pollinations')
        except Exception as e:
            print('  ⚠️ Gemini falló → Pollinations:', e)
    r = requests.post('https://text.pollinations.ai/openai', json={
        'model': 'openai',
        'messages': [{'role': 'system', 'content': 'Responde SOLO con JSON valido, sin markdown ni explicaciones.'},
                     {'role': 'user', 'content': prompt}]}, timeout=180)
    r.raise_for_status()
    return _json(r.json()['choices'][0]['message']['content'])

def imagen_gratis(prompt, destino, w, h, seed=42):
    url = ('https://image.pollinations.ai/prompt/' + urllib.parse.quote(prompt[:1200])
           + f'?width={w}&height={h}&nologo=true&seed={seed}&model=flux')
    ultimo = None
    for intento in range(3):
        try:
            r = requests.get(url, timeout=300); r.raise_for_status()
            destino.write_bytes(r.content); return
        except Exception as e:
            ultimo = e; print(f'  reintento imagen {intento+1}/3:', e); time.sleep(5)
    raise HTTPException(502, f'Pollinations no respondió: {ultimo}')

def ffmpeg(*args):
    p = subprocess.run(['ffmpeg', '-y', *args], capture_output=True, text=True)
    if p.returncode != 0: raise HTTPException(500, 'FFmpeg: ' + p.stderr[-300:])

# ---------------- ENDPOINTS ----------------
@app.get('/api/health')
def health():
    return {'ok': True, 'gpu': torch.cuda.is_available(), 'modelo': 'LTX-Video', 'version': 'arreglada-1.0'}

@app.post('/api/proyecto')
def nuevo(audio: UploadFile, letra: str = Form(''), formato: str = Form('16:9'), x_token: str = Header(default='')):
    auth(x_token)
    pid = uuid.uuid4().hex[:12]
    c = PROY / pid; c.mkdir()
    ap = c / ('audio' + Path(audio.filename or '.mp3').suffix)
    ap.write_bytes(audio.file.read())
    bpm, dur = 0, 0.0
    try:
        import librosa
        y, sr = librosa.load(str(ap), sr=22050, mono=True)
        dur = float(librosa.get_duration(y=y, sr=sr))
        t, _ = librosa.beat.beat_track(y=y, sr=sr)
        bpm = round(float(t[0] if hasattr(t, '__len__') else t))
    except Exception as e:
        print('  análisis BPM falló (sigo sin él):', e)
    if dur == 0: dur = 180.0
    guardar(pid, {'id': pid, 'letra': letra, 'formato': formato, 'bpm': bpm,
                  'duracion': round(dur, 1), 'audio': ap.name, 'conceptos': [],
                  'concepto': None, 'personaje': None, 'escenas': [], 'final': None})
    return {'id': pid, 'bpm': bpm, 'duracion': round(dur, 1), 'formato': formato}

@app.post('/api/proyecto/{pid}/conceptos')
def conceptos(pid: str, x_token: str = Header(default='')):
    auth(x_token); est = cargar(pid)
    d = ia_texto(
        'Eres un director de videoclips premiado. Letra de la canción:\n\"\"\"'
        + est['letra'][:3000] + '\"\"\"\n'
        'Propón 3 conceptos visuales MUY distintos (fotorrealista, anime, cyberpunk, noir, onírico...).\n'
        'Devuelve JSON: {"conceptos":[{"nombre":"...","descripcion":"2 frases de narrativa visual",'
        '"paleta":"colores","personaje":"descripción física del protagonista para generar su imagen"}]}')
    est['conceptos'] = d['conceptos'][:3]; guardar(pid, est)
    return {'conceptos': est['conceptos']}

@app.post('/api/proyecto/{pid}/personaje')
def personaje(pid: str, body: dict, x_token: str = Header(default='')):
    auth(x_token); est = cargar(pid)
    idx = body.get('concepto')
    estilo = est['conceptos'][idx]['nombre'] if isinstance(idx, int) and est['conceptos'] else 'cinematográfico'
    desc = body.get('descripcion', 'cantante protagonista')
    prompt = (f'character sheet, three views (front, profile, full body) of {desc}, '
              f'{estilo} style, consistent character, neutral background, detailed, high quality')
    w, h = (1216, 704) if est['formato'] == '16:9' else (704, 1216)
    d = carpeta(pid) / 'personaje.png'
    imagen_gratis(prompt, d, w, h, seed=7)
    est['personaje'] = {'descripcion': desc, 'archivo': 'personaje.png'}; guardar(pid, est)
    return {'url': f'/api/proyecto/{pid}/archivo/personaje.png'}

@app.post('/api/proyecto/{pid}/storyboard')
def storyboard(pid: str, body: dict, x_token: str = Header(default='')):
    auth(x_token); est = cargar(pid)
    idx = body.get('concepto')
    if isinstance(idx, int): est['concepto'] = idx
    con = est['conceptos'][est['concepto']] if est['conceptos'] and est['concepto'] is not None else {}
    n = max(4, round(est['duracion'] / 5)); ds = round(est['duracion'] / n, 1)
    d = ia_texto(
        f'Eres director de videoclips. Canción de {est["duracion"]}s, {est["bpm"]} BPM.\n'
        f'Concepto visual: {json.dumps(con, ensure_ascii=False)}\n'
        f'Personaje: {(est.get("personaje") or {}).get("descripcion", "")}\n'
        'Letra:\n\"\"\"' + est['letra'][:3000] + '\"\"\"\n'
        f'Crea EXACTAMENTE {n} escenas de {ds}s en orden (intro, versos, coros...). lipsync SIEMPRE false.\n'
        f'Cada prompt visual, detallado, en inglés, con el personaje y el estilo "{con.get("nombre", "")}".\n'
        'JSON: {"escenas":[{"seccion":"verso/chorus/...","prompt":"...","lipsync":false}]}')
    est['escenas'] = [{'n': i, 'seccion': e.get('seccion', ''), 'prompt': e['prompt'],
                       'duracion': ds, 'lipsync': False, 'imagen': None, 'video': None}
                      for i, e in enumerate(d['escenas'][:n])]
    guardar(pid, est)
    return {'escenas': est['escenas']}

@app.post('/api/proyecto/{pid}/escenas/{i}/imagen')
def escena_imagen(pid: str, i: int, body: dict, x_token: str = Header(default='')):
    auth(x_token); est = cargar(pid); esc = est['escenas'][i]
    pd = (est.get('personaje') or {}).get('descripcion', '')
    prompt = body.get('prompt') or esc['prompt']
    full = f'{prompt}. Main character: {pd}. Same character, cinematic still frame.'
    w, h = (1216, 704) if est['formato'] == '16:9' else (704, 1216)
    d = carpeta(pid) / f'escena_{i}.png'
    imagen_gratis(full, d, w, h, seed=100 + i)
    esc['prompt'] = prompt; esc['imagen'] = d.name; guardar(pid, est)
    return {'url': f'/api/proyecto/{pid}/archivo/{d.name}'}

def clip_kenburns(img_path, out_mp4, duracion, w, h):
    fps=24; total=int(duracion*fps); step=0.12/max(total,1)
    vf=(f"scale={w*2}:{h*2}:force_original_aspect_ratio=increase,crop={w*2}:{h*2},"
        f"zoompan=z='min(zoom+{step:.6f},1.12)':x='iw/2-(iw/zoom/2)':y='ih/2-(ih/zoom/2)':"
        f"d={total}:s={w}x{h}:fps={fps},format=yuv420p")
    ffmpeg("-loop","1","-i",str(img_path),"-vf",vf,"-t",f"{duracion}",
           "-c:v","libx264","-preset","veryfast","-r",str(fps),str(out_mp4))

@app.post('/api/proyecto/{pid}/escenas/{i}/video')
def escena_video(pid: str, i: int, body: dict, x_token: str = Header(default='')):
    auth(x_token); est = cargar(pid); esc = est['escenas'][i]
    imgp = carpeta(pid) / (esc.get('imagen') or f'escena_{i}.png')
    if not imgp.exists(): raise HTTPException(400, 'Primero genera la imagen de la escena')
    seg = max(2.0, min(float(body.get('duracion') or esc['duracion']), 8.0))
    d = carpeta(pid) / f'escena_{i}.mp4'
    if MODO_VIDEO == "ltx" and pipe is not None:
        prompt = (body.get('prompt') or esc['prompt']) + ', cinematic, smooth motion, high quality'
        fps = 24; frames = int(seg * fps) // 8 * 8 + 1
        img = Image.open(imgp).convert('RGB'); img.thumbnail((640, 640))
        print(f'🎬 [LTX] Escena {i+1}: {frames} frames...')
        out = pipe(image=img, prompt=prompt, num_frames=frames, frame_rate=fps,
                   num_inference_steps=20, guidance_scale=3.0).frames[0]
        export_to_video(out, str(d), fps=fps)
        del out, img; gc.collect(); torch.cuda.empty_cache()
    else:
        w,h = (576,1024) if est.get('formato')=='9:16' else (1024,576)
        print(f'🎬 [Ken Burns] Escena {i+1}: zoom {w}x{h} (FFmpeg, sin RAM)...')
        clip_kenburns(imgp, d, seg, w, h)
    esc['video'] = d.name; guardar(pid, est)
    print(f'✅ Escena {i+1} lista')
    return {'url': f'/api/proyecto/{pid}/archivo/{d.name}'}

@app.post('/api/proyecto/{pid}/escenas/{i}/lipsync')
def lipsync_noop(pid: str, i: int, x_token: str = Header(default='')):
    auth(x_token); est = cargar(pid)
    v = est['escenas'][i].get('video')
    if not v: raise HTTPException(400, 'Primero genera el video de la escena')
    return {'url': f'/api/proyecto/{pid}/archivo/{v}', 'nota': 'lipsync no incluido en esta versión'}

@app.post('/api/proyecto/{pid}/ensamblar')
def ensamblar(pid: str, x_token: str = Header(default='')):
    auth(x_token); est = cargar(pid); c = carpeta(pid)
    clips = [c / e['video'] for e in est['escenas'] if e.get('video')]
    if not clips: raise HTTPException(400, 'No hay escenas generadas')
    lista = c / 'lista.txt'
    lista.write_text(''.join(f"file '{x}'\n" for x in clips))
    ffmpeg('-f', 'concat', '-safe', '0', '-i', str(lista),
           '-c:v', 'libx264', '-pix_fmt', 'yuv420p', '-an', str(c / 'sin_audio.mp4'))
    ffmpeg('-i', str(c / 'sin_audio.mp4'), '-i', str(c / est['audio']),
           '-c:v', 'copy', '-c:a', 'aac', '-shortest', str(c / 'videoclip_final.mp4'))
    est['final'] = 'videoclip_final.mp4'; guardar(pid, est)
    return {'url': f'/api/proyecto/{pid}/archivo/videoclip_final.mp4'}

@app.get('/api/proyecto/{pid}')
def estado(pid: str, x_token: str = Header(default='')):
    auth(x_token); return cargar(pid)

@app.get('/api/proyecto/{pid}/archivo/{nombre}')
def archivo(pid: str, nombre: str):
    # PÚBLICO a propósito: las etiquetas <img>/<video> del Lanzador NO pueden enviar el token
    if '/' in nombre or '..' in nombre: raise HTTPException(400, 'nombre inválido')
    p = carpeta(pid) / nombre
    if not p.exists(): raise HTTPException(404, 'archivo no existe')
    return FileResponse(str(p))

nest_asyncio.apply()
threading.Thread(target=lambda: uvicorn.run(app, host='0.0.0.0', port=8080), daemon=True).start()
time.sleep(5)
print('✅ SERVIDOR COMPLETO corriendo en puerto 8080')



In [ ]:
# 🌐 CELDA 5/5: Túnel público + URL GRANDE PARA COPIAR
import subprocess, re, time

print('⏳ Creando túnel público...')
proc = subprocess.Popen(['/content/cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8080'],
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
url = None
for _ in range(120):
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', proc.stdout.readline())
    if m:
        url = m.group(0); break

if not url:
    print('❌ No se pudo crear el túnel. Vuelve a ejecutar SOLO esta celda.')
else:
    import requests as rq
    try:
        print('Test health:', rq.get(url + '/api/health', timeout=30).json())
    except Exception as e:
        print('(health tardará unos segundos en responder:', e, ')')
    print()
    print('=' * 64)
    print()
    print('   🎬 ¡TODO LISTO!  Copia esta URL 👇')
    print()
    print('   🔗 ', url)
    print()
    print('   Pégala en el Lanzador → campo "URL del Orquestador"')
    print('   Token: el mismo de la CELDA 2')
    print()
    print('   ⚠️ NO cierres esta pestaña mientras generas videoclips.')
    print()
    print('=' * 64)
    while True:
        time.sleep(60)